In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

def fast_unzip(zip_path, extract_to):
    print(f'Đang giải nén {zip_path}...')
    if not os.path.exists(extract_to):
        os.makedirs(extract_to)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f'Hoàn thành giải nén vào {extract_to}')

# Đường dẫn tệp
train_zip = "/content/drive/My Drive/AIO_Homework/dence representation/data/data_train.zip"
test_zip = "/content/drive/My Drive/AIO_Homework/dence representation/data/data_test.zip"
dataset_path = "/content/dataset/"

# Chạy giải nén
if os.path.exists(train_zip):
    fast_unzip(train_zip, dataset_path)
else:
    print('Không tìm thấy tệp zip train. Kiểm tra lại đường dẫn!')

if os.path.exists(test_zip):
    fast_unzip(test_zip, dataset_path)
else:
    print('Không tìm thấy tệp zip test.')

Đang giải nén /content/drive/My Drive/AIO_Homework/dence representation/data/data_train.zip...
Hoàn thành giải nén vào /content/dataset/
Đang giải nén /content/drive/My Drive/AIO_Homework/dence representation/data/data_test.zip...
Hoàn thành giải nén vào /content/dataset/


In [ ]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from tqdm import tqdm

In [ ]:
def load_corpus_with_labels(base_directory):
    corpus = []
    labels = []
    for root, dirs, files in os.walk(base_directory):
        # Determine label based on directory name
        if 'pos' in root.split(os.sep):
            label = 1  # Positive sentiment
        elif 'neg' in root.split(os.sep):
            label = 0  # Negative sentiment
        else:
            # If neither pos nor neg, skip this directory
            continue

        for filename in files:
            if filename.endswith(".txt"):
                filepath = os.path.join(root, filename)
                with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                    corpus.append(f.read())
                    labels.append(label)
    return corpus, labels

TRAIN_DIR = "/content/dataset/data_train/train"
VAL_DIR   = "/content/dataset/data_train/test"
TEST_DIR  = "/content/dataset/data_test/test"

train_corpus, train_labels = load_corpus_with_labels(TRAIN_DIR)
val_corpus, val_labels     = load_corpus_with_labels(VAL_DIR)
test_corpus, test_labels   = load_corpus_with_labels(TEST_DIR)

print(f"Train docs: {len(train_corpus)}, Train labels: {len(train_labels)}")
print(f"Val docs  : {len(val_corpus)}, Val labels: {len(val_labels)}")
print(f"Test docs : {len(test_corpus)}, Test labels: {len(test_labels)}")

Train docs: 30000, Train labels: 30000
Val docs  : 10000, Val labels: 10000
Test docs : 10000, Test labels: 10000


In [ ]:
import re
from collections import Counter

def preprocess_text(text):
    text = text.lower()  # Chuyển đổi sang chữ thường
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Loại bỏ ký tự đặc biệt
    tokens = text.split()  # Tách từ
    return tokens

train_tokens = [preprocess_text(doc) for doc in train_corpus]
val_tokens   = [preprocess_text(doc) for doc in val_corpus]
test_tokens  = [preprocess_text(doc) for doc in test_corpus]

print(f"First 10 tokens of first training document: {train_tokens[0][:10]}")

First 10 tokens of first training document: ['rt', 'thch', 'n', 'y', 'khnggian', 'rng', 'sangtrng', 'v', 'lchs', 'trong']


In [ ]:
from collections import Counter

def build_vocab(tokenized_corpus, min_freq=5):
    word_counts = Counter()
    for doc_tokens in tokenized_corpus:
        word_counts.update(doc_tokens)

    # Lọc các từ có tần suất thấp
    vocab = [word for word, count in word_counts.items() if count >= min_freq]
    word_to_idx = {word: idx + 1 for idx, word in enumerate(vocab)} # idx 0 cho PAD
    word_to_idx['<unk>'] = 0 # Từ không biết
    return word_to_idx, vocab, word_counts

# Xây dựng từ vựng từ tập huấn luyện
word_to_idx, vocab, word_counts = build_vocab(train_tokens)

print(f"Kích thước từ vựng: {len(vocab)}")
print(f"Top 10 từ phổ biến nhất: {word_counts.most_common(10)}")

Kích thước từ vựng: 7190
Top 10 từ phổ biến nhất: [('c', 76223), ('n', 75369), ('v', 56680), ('mnh', 49163), ('th', 44515), ('l', 37715), ('ch', 35257), ('qun', 34444), ('m', 33143), ('i', 29833)]


In [ ]:
def get_ngrams(word, n):
    word = '<' + word + '>'
    return [word[i:i+n] for i in range(len(word) - n + 1)]

def build_subword_vocab(word_to_idx, n_gram_min=3, n_gram_max=5):
    subword_to_idx = {}
    idx = 1 # Bắt đầu từ 1, 0 là cho PAD/UNK subword
    for word in word_to_idx.keys():
        if word == '<unk>': # Bỏ qua <unk> vì nó không có subword
            continue
        for n in range(n_gram_min, n_gram_max + 1):
            ngrams = get_ngrams(word, n)
            for ng in ngrams:
                if ng not in subword_to_idx:
                    subword_to_idx[ng] = idx
                    idx += 1
    subword_to_idx['<unk_sub>'] = 0
    return subword_to_idx

subword_to_idx = build_subword_vocab(word_to_idx)
print(f"Kích thước từ vựng subword: {len(subword_to_idx)}")
print(f"Một vài subword ví dụ: {list(subword_to_idx.keys())[:20]}")

Kích thước từ vựng subword: 33238
Một vài subword ví dụ: ['<rt', 'rt>', '<rt>', '<th', 'thc', 'hch', 'ch>', '<thc', 'thch', 'hch>', '<thch', 'thch>', '<n>', '<y>', '<kh', 'khn', 'hng', 'ngg', 'ggi', 'gia']


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def tokens_to_indices(tokenized_corpus, word_to_idx, max_seq_len):
    indexed_corpus = []
    for doc_tokens in tokenized_corpus:
        indices = [word_to_idx.get(token, word_to_idx['<unk>']) for token in doc_tokens]
        # Pad hoặc cắt bớt chuỗi để có độ dài cố định
        if len(indices) < max_seq_len:
            indices += [0] * (max_seq_len - len(indices)) # Pad với 0 (PAD_idx)
        else:
            indices = indices[:max_seq_len]
        indexed_corpus.append(indices)
    return torch.tensor(indexed_corpus, dtype=torch.long)

def subword_to_indices(tokenized_corpus, subword_to_idx, n_gram_min, n_gram_max, max_subword_per_word, max_seq_len):
    indexed_corpus_subword = []
    for doc_tokens in tokenized_corpus:
        doc_subword_indices = []
        for token in doc_tokens:
            word_subword_indices = []
            for n in range(n_gram_min, n_gram_max + 1):
                ngrams = get_ngrams(token, n)
                word_subword_indices.extend([subword_to_idx.get(ng, subword_to_idx['<unk_sub>']) for ng in ngrams])
            # Pad hoặc cắt bớt subword của từng từ
            if len(word_subword_indices) < max_subword_per_word:
                word_subword_indices += [0] * (max_subword_per_word - len(word_subword_indices))
            else:
                word_subword_indices = word_subword_indices[:max_subword_per_word]
            doc_subword_indices.append(word_subword_indices)

        # Pad hoặc cắt bớt chuỗi các từ
        if len(doc_subword_indices) < max_seq_len:
            doc_subword_indices += [[0] * max_subword_per_word] * (max_seq_len - len(doc_subword_indices))
        else:
            doc_subword_indices = doc_subword_indices[:max_seq_len]

        indexed_corpus_subword.append(doc_subword_indices)

    return torch.tensor(indexed_corpus_subword, dtype=torch.long)

# Xác định độ dài tối đa của chuỗi và số subword tối đa cho mỗi từ
MAX_SEQ_LEN = 100  # Ví dụ: 100 từ mỗi tài liệu
MAX_SUBWORD_PER_WORD = 20 # Ví dụ: 20 subword mỗi từ
N_GRAM_MIN = 3
N_GRAM_MAX = 5

# Chuyển đổi token và subword thành chỉ số
train_indexed_words = tokens_to_indices(train_tokens, word_to_idx, MAX_SEQ_LEN)
val_indexed_words   = tokens_to_indices(val_tokens, word_to_idx, MAX_SEQ_LEN)
test_indexed_words  = tokens_to_indices(test_tokens, word_to_idx, MAX_SEQ_LEN)

train_indexed_subwords = subword_to_indices(train_tokens, subword_to_idx, N_GRAM_MIN, N_GRAM_MAX, MAX_SUBWORD_PER_WORD, MAX_SEQ_LEN)
val_indexed_subwords   = subword_to_indices(val_tokens, subword_to_idx, N_GRAM_MIN, N_GRAM_MAX, MAX_SUBWORD_PER_WORD, MAX_SEQ_LEN)
test_indexed_subwords  = subword_to_indices(test_tokens, subword_to_idx, N_GRAM_MIN, N_GRAM_MAX, MAX_SUBWORD_PER_WORD, MAX_SEQ_LEN)

# Chuyển đổi nhãn thành tensor
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)
val_labels_tensor   = torch.tensor(val_labels, dtype=torch.long)
test_labels_tensor  = torch.tensor(test_labels, dtype=torch.long)

# Tạo TensorDataset
train_dataset = TensorDataset(train_indexed_words, train_indexed_subwords, train_labels_tensor)
val_dataset   = TensorDataset(val_indexed_words, val_indexed_subwords, val_labels_tensor)
test_dataset  = TensorDataset(test_indexed_words, test_indexed_subwords, test_labels_tensor)

# Tạo DataLoader
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Kích thước train_indexed_words: {train_indexed_words.shape}")
print(f"Kích thước train_indexed_subwords: {train_indexed_subwords.shape}")
print(f"Kích thước train_labels_tensor: {train_labels_tensor.shape}")
print(f"Số batch trong train_loader: {len(train_loader)}")

Kích thước train_indexed_words: torch.Size([30000, 100])
Kích thước train_indexed_subwords: torch.Size([30000, 100, 20])
Kích thước train_labels_tensor: torch.Size([30000])
Số batch trong train_loader: 469


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FastText(nn.Module):
    def __init__(self, vocab_size, subword_vocab_size, embedding_dim, num_classes):
        super(FastText, self).__init__()
        self.embedding_dim = embedding_dim

        # Word embeddings
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # Subword embeddings
        # Sử dụng subword_vocab_size + 1 vì 0 được dùng cho PAD/UNK subword
        self.subword_embeddings = nn.Embedding(subword_vocab_size + 1, embedding_dim, padding_idx=0)

        # Output linear layer
        self.fc = nn.Linear(embedding_dim, num_classes)

    def forward(self, words, subwords):
        # Word embeddings
        word_embeds = self.word_embeddings(words)

        # Subword embeddings
        # subwords: (batch_size, max_seq_len, max_subword_per_word)
        batch_size, max_seq_len, max_subword_per_word = subwords.shape

        # Reshape subwords for embedding lookup: (batch_size * max_seq_len * max_subword_per_word)
        subwords_flat = subwords.view(-1)

        # Get subword embeddings: (batch_size * max_seq_len * max_subword_per_word, embedding_dim)
        subword_embeds_flat = self.subword_embeddings(subwords_flat)

        # Reshape back to (batch_size, max_seq_len, max_subword_per_word, embedding_dim)
        subword_embeds_reshaped = subword_embeds_flat.view(batch_size, max_seq_len, max_subword_per_word, self.embedding_dim)

        # Average subword embeddings for each word
        # Ignore padding (value 0) in subword averaging
        # Create a mask for non-padding subwords using the original subwords tensor
        subword_mask = (subwords != 0).float().unsqueeze(-1) # (batch_size, max_seq_len, max_subword_per_word, 1)

        # Apply mask to subword embeddings and sum
        masked_subword_embeds = subword_embeds_reshaped * subword_mask
        sum_subword_embeds = masked_subword_embeds.sum(dim=2) # (batch_size, max_seq_len, embedding_dim)

        # Count non-padding subwords for averaging
        subword_counts = subword_mask.sum(dim=2).clamp(min=1) # (batch_size, max_seq_len, 1)
        avg_subword_embeds = sum_subword_embeds / subword_counts # (batch_size, max_seq_len, embedding_dim)

        # Kết hợp word embeddings và subword embeddings (ví dụ: cộng hoặc trung bình)
        # Ở đây tôi sẽ cộng chúng lại, bạn có thể thử các cách khác
        combined_embeds = word_embeds + avg_subword_embeds

        # Trung bình cộng các embedding của các từ trong mỗi tài liệu
        # Giữ nguyên padding (0) cho word_embeds để không ảnh hưởng đến trung bình cộng
        # word_embeds: (batch_size, max_seq_len, embedding_dim)
        # sum all embeddings and divide by the number of non-zero embeddings
        # Create a mask for non-padding words
        word_mask = (words != 0).float().unsqueeze(-1) # (batch_size, max_seq_len, 1)

        # Apply mask to combined embeddings and sum
        masked_combined_embeds = combined_embeds * word_mask
        sum_combined_embeds = masked_combined_embeds.sum(dim=1) # (batch_size, embedding_dim)

        # Count non-padding words for averaging
        word_counts_for_avg = word_mask.sum(dim=1).clamp(min=1) # (batch_size, 1)
        avg_doc_embeds = sum_combined_embeds / word_counts_for_avg # (batch_size, embedding_dim)

        # Dự đoán phân loại
        logits = self.fc(avg_doc_embeds)
        return logits


# Khởi tạo mô hình
VOCAB_SIZE = len(word_to_idx)
SUBWORD_VOCAB_SIZE = len(subword_to_idx)
EMBEDDING_DIM = 100 # Kích thước vector nhúng
NUM_CLASSES = 2 # 0 cho tiêu cực, 1 cho tích cực

model = FastText(VOCAB_SIZE, SUBWORD_VOCAB_SIZE, EMBEDDING_DIM, NUM_CLASSES)

print(model)

FastText(
  (word_embeddings): Embedding(7191, 100, padding_idx=0)
  (subword_embeddings): Embedding(33239, 100, padding_idx=0)
  (fc): Linear(in_features=100, out_features=2, bias=True)
)


In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Thiết lập thiết bị (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Hàm mất mát và Tối ưu hóa
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, patience=5, min_delta=0.001):
    best_val_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        model.train() # Chế độ huấn luyện
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        for words, subwords, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
            words, subwords, labels = words.to(device), subwords.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(words, subwords)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_accuracy = correct_predictions / total_samples

        val_loss, val_accuracy = evaluate_model(model, val_loader, criterion)

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

        # Early stopping logic
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {patience} epochs with no improvement.")
                break

def evaluate_model(model, data_loader, criterion):
    model.eval() # Chế độ đánh giá
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for words, subwords, labels in data_loader:
            words, subwords, labels = words.to(device), subwords.to(device), labels.to(device)

            outputs = model(words, subwords)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    avg_loss = running_loss / len(data_loader)
    accuracy = correct_predictions / total_samples
    return avg_loss, accuracy

# Bắt đầu huấn luyện
NUM_EPOCHS = 50 # Bạn có thể điều chỉnh số epoch
train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=NUM_EPOCHS)

# Đánh giá trên tập kiểm tra
test_loss, test_accuracy = evaluate_model(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

Epoch 1/50 - Training: 100%|██████████| 469/469 [00:03<00:00, 143.50it/s]


Epoch 1/50, Train Loss: 0.3060, Train Acc: 0.8814, Val Loss: 0.3766, Val Acc: 0.8508


Epoch 2/50 - Training: 100%|██████████| 469/469 [00:02<00:00, 163.51it/s]


Epoch 2/50, Train Loss: 0.2976, Train Acc: 0.8853, Val Loss: 0.3781, Val Acc: 0.8494


Epoch 3/50 - Training: 100%|██████████| 469/469 [00:02<00:00, 161.92it/s]


...

Epoch 4/50, Train Loss: 0.2845, Train Acc: 0.8913, Val Loss: 0.3856, Val Acc: 0.8483


Epoch 5/50 - Training: 100%|██████████| 469/469 [00:03<00:00, 147.98it/s]


Epoch 5/50, Train Loss: 0.2787, Train Acc: 0.8942, Val Loss: 0.3894, Val Acc: 0.8461


Epoch 6/50 - Training: 100%|██████████| 469/469 [00:02<00:00, 159.51it/s]


Epoch 6/50, Train Loss: 0.2736, Train Acc: 0.8960, Val Loss: 0.3971, Val Acc: 0.8421
Early stopping triggered after 5 epochs with no improvement.
Test Loss: 0.3710, Test Accuracy: 0.8524
